# Ejercicio 2

###  Python: API APOD de la NASA

**Preguntas del equipo:**
* ¿Qué tipo de parámetros admite la solicitud?
* ¿Qué tipo respuestas se pueden obtener con cada solicitud?
* ¿Qué tipo de restricciones tiene la API?

**Preguntas de cada miembro del equipo:**

1. Para las búsquedas de tu palabra, ¿cuántos resultados obtuviste?
2. Para las búsquedas de tu palabra, ¿en qué rangos de fechas se introdujo el recurso?
3. Para las búsquedas de tu palabra, ¿cuáles son los "media_type" más comunes?
4. Para las búsquedas de tu palabra, ¿quiénes son los autores o instituciones propietaria de los derechos (i.e. el copyright)?





1. ¿Qué tipo de parámetros admite la solicitud?

Las consultas a la API admiten los siguintes parametros:

* **api_key**: la llave para acceder a la API, puede ser una llave publica (imita la cantidad de consultas) o una llave personal generada de forma gratuita en el sitio
* **date**: Una cadena de texto en formato AAAA-MM-DD que indica la fecha de la imagen APOD a consultar (ejemplo: 2002-12-03). En caso de no intrudicir una fecha se utiliza la fecha actual al momento de la consulta.
* **start_date**: Una cadena de texto en formato AAAA-MM-DD que indica el inicio de un rango de fechas, se usa en conjuntoa end_dete para crear un intervalo de tiempo.
* **end_date**: Una cadena de texto en formato AAAA-MM-DD que indica el fin de un rango de fechas, se usa en conjuntoa start_dete para crear un intervalo de tiempo.
* **concept_tags**: Un valor booleano True o False que indica si las etiquetas de concept deben devolverse junto con el resto de la respuesta.
* **hd** : Un valor booleano True o False que indica si se deben devolver imágenes de alta resolución.
* **count** : Un número entero positivo, que si se especifica, se devolverán imágenes elegidas aleatoriamente en una matriz JSON.
* **thumbs**: Un valor booleano True o False que indica si la API debe devolver la URL de la imagen en miniatura para los archivos de vídeo.

Pese a que la API APOD tiene todos estos parámetros para las consultas, no todos se pueden usar al mimso timepo y existenen ciertas restricciones para las consultas, dichas restricciones se comentaran mas adelante.

2. ¿Qué tipo respuestas se pueden obtener con cada solicitud?

La API, generalmente devolvera un diccionario o una lista de diccionaros, segun sea el tipo de solicitud.

Si en nuestra solicitud usamos el parametro date, es decir consultamos un unico día la api devuelve un diccionario JSON, cuando en nuestra consulta usamos los parametros start_date y end_date o el parametro count, la API nos devolverá una lista de diccionaros.

Tambien podemos tener respuestas de error

3. ¿Qué tipo de restricciones tiene la API?

La API APOD cuenta con 2 tipos de restricciones

1. **Restricció en cantidad de consultas**

Usando una API KEY de prueba se tienen límites por IP muy bajos:

* 30 consultas por hora
* 50 consultas por día

Usando una API KEY propia

* Se obtiene un limite de  1000 requests por hora

2. **Restriccion en estructura de consulta**

La principal restriccion al momento de hacer nuestras consultas son con las fechas

* **Primer fecha**: APOD solo tiene registros desde: 1995-06-16, por lo que la consulta debe hacerce posterior a esta fecha y no mayor a la fecha actual.
* **Convinación de parametros**: No es posible usar los siguintes parametos juntos : date, start_date, end_date o count, la  unica conbinación posible es start_date y end_date para consultar intervalos de fechas


**Palabras correspondientes al número de la primera letra del apellido paterno de cada integrante del equipo:**

*   Aguilera Yáñez Mariana --> ***gravitational***
*   Alvarado Arce Axel Eduardo --> ***retrograde***
*   Canseco Galván Tanivet --> ***supernova***
*   Garcia Soto Kevin --> ***nebula***
*   Hernandez Mareles Francisco Javier --> ***planetary***
*   Rendon Cardoso Karla --> ***Geminids***
*   Suarez Urbano David Hiram --> ***Einstein***


Pimero contruiremos el codigo para acceder a los datos medinate la API APOD de la NASA

In [ ]:
import requests
import pandas as pd
import time
from datetime import datetime, timedelta

In [ ]:
def get_apod_dataframe(api_key, start_date, end_date):
    url = "https://api.nasa.gov/planetary/apod"

    start = datetime.strptime(start_date, "%Y-%m-%d")
    end = datetime.strptime(end_date, "%Y-%m-%d")

    # Para evitar errores de la API usaremos maximo bloques de 100 dias
    a = timedelta(days=100)
    inicio = start

    all_data = []

    while inicio <= end:
        chunk_end = min(inicio + a, end)

        params = {
            "api_key": api_key,
            "start_date": inicio.strftime("%Y-%m-%d"),
            "end_date": chunk_end.strftime("%Y-%m-%d"),
            "thumbs": True  # para obtener thumbnail en videos
        }

        response = requests.get(url, params=params)

        if response.status_code == 200:
            data = response.json()

            # para evitar errores de lista
            if isinstance(data, dict):
                data = [data]

            for item in data:
                all_data.append({
                    "resource": item.get("resource"),
                    "concept_tags": item.get("concept_tags"),
                    "title": item.get("title"),
                    "date": item.get("date"),
                    "url": item.get("url"),
                    "hdurl": item.get("hdurl"),
                    "media_type": item.get("media_type"),
                    "explanation": item.get("explanation"),
                    "concepts": item.get("concepts"),
                    "thumbnail_url": item.get("thumbnail_url"),
                    "copyright": item.get("copyright"),
                    "service_version": item.get("service_version")
                })

        else:
            print(f"Error {response.status_code} en rango {inicio} - {chunk_end}")

        # pausa para evitar que la api nos bloque
        time.sleep(1)

        inicio = chunk_end + timedelta(days=1)

    # Convertir a DataFrame
    df = pd.DataFrame(all_data)

    return df

In [ ]:
df = get_apod_dataframe(
    api_key="S1ba65tHYH5KWWPUVwYfnGs3wpdXYf6TDMBQSQqk",
    start_date="2003-04-13",
    end_date="2003-05-13"
)

df

,resource,concept_tags,title,date,url,hdurl,media_type,explanation,concepts,thumbnail_url,copyright,service_version
0,None,None,NGC 1365: A Nearby Barred Spiral Galaxy,2003-04-13,https://apod.nasa.gov/apod/image/0304/ngc1365_...,https://apod.nasa.gov/apod/image/0304/ngc1365_...,image,Many spiral galaxies have bars across their ce...,None,None,None,v1
1,None,None,A Gamma Ray Burst - Supernova Connection,2003-04-14,https://apod.nasa.gov/apod/image/0304/ngc3184_...,https://apod.nasa.gov/apod/image/0304/ngc3184_...,image,New evidence has emerged that a mysterious typ...,None,None,None,v1
2,None,None,A Crescent Nebula Star Field,2003-04-15,https://apod.nasa.gov/apod/image/0304/crescent...,https://apod.nasa.gov/apod/image/0304/crescent...,image,What caused the Crescent Nebula? Looking like...,None,None,T. A. Rector,v1
3,None,None,Magma Bubbles from Mt. Etna,2003-04-16,https://apod.nasa.gov/apod/image/0304/etnabubb...,https://apod.nasa.gov/apod/image/0304/etnabubb...,image,Mt. Etna erupted spectacularly in 2001 June. ...,None,None,Stromboli online,v1
4,None,None,M106 in Canes Venatici,2003-04-17,https://apod.nasa.gov/apod/image/0304/m106_blo...,https://apod.nasa.gov/apod/image/0304/m106_blo...,image,Close to the Great Bear (Ursa Major) and surro...,None,None,None,v1
5,None,None,Double Eruptive Prominences,2003-04-18,https://apod.nasa.gov/apod/image/0304/doublepr...,https://apod.nasa.gov/apod/image/0304/doublepr...,image,Lofted over the Sun on looping magnetic fields...,None,None,None,v1
6,None,None,Spiral Galaxy In Centaurus,2003-04-19,https://apod.nasa.gov/apod/image/0304/eso269_v...,https://apod.nasa.gov/apod/image/0304/eso269_v...,image,"Centaurus, the Centaur, is one of the most str...",None,None,None,v1
7,None,None,The Gum Nebula Supernova Remnant,2003-04-20,https://apod.nasa.gov/apod/image/0304/gum_glea...,https://apod.nasa.gov/apod/image/0304/gum_glea...,image,Because the Gum Nebula is the closest supernov...,None,None,John Gleason (Celestial Images),v1
8,None,None,A Halo Around the Moon,2003-04-21,https://apod.nasa.gov/apod/image/0304/moonhalo...,https://apod.nasa.gov/apod/image/0304/moonhalo...,image,Tomorrow's picture: North Mars < | Archive | ...,None,None,Sarah McKay,v1
9,None,None,Springtime on Mars,2003-04-22,https://apod.nasa.gov/apod/image/0304/marsnort...,https://apod.nasa.gov/apod/image/0304/marsnort...,image,"Vast canyons, towering volcanoes, sprawling fi...",None,None,None,v1


In [ ]:
import pandas as pd

def analizar_apod(df, palabra_clave):
    # Solo para asegurar que sienpre sea un str
    df["explanation"] = df["explanation"].astype(str)

    # Filtramos por palabra
    df_filtrado = df[df["explanation"].str.contains(palabra_clave, case=False, na=False)]

    # Numero de resultados
    total_resultados = len(df_filtrado)

    # Media_type
    media_types = df_filtrado["media_type"].value_counts()

    # Copyrights
    copyrights = (
        df_filtrado["copyright"]
        .dropna()
        .value_counts()
    )

    return {
        "total": total_resultados,
        "media_types": media_types,
        "copyrights": copyrights,
        "df_filtrado": df_filtrado
    }

In [ ]:
resultado = analizar_apod(df, "Nebula")

*   Aguilera Yáñez Mariana --> ***gravitational***

In [ ]:
df_Mariana = get_apod_dataframe(
    api_key="S1ba65tHYH5KWWPUVwYfnGs3wpdXYf6TDMBQSQqk",
    start_date="2010-08-20",
    end_date="2010-11-20"
)

resultado1 = analizar_apod(df_Mariana, "gravitational")

In [ ]:
import pandas as pd

# Convertir copyrights a lista de texto
lista_copyrights = ", ".join(
    [f" {nombre}: {cantidad}"
     for nombre, cantidad in resultado1['copyrights'].items()]
)


# Diccionario con los datos
datos_resumen = {
    "Concepto": ["Alumno","Intervalo de fechas", "Palabra buscada",
                 "Total de Resultados", "Media Type Imagen", "Media Type Videos",
                  "Copyrights"],
    "Valor": [ "Aguilera Yañez Mariana",
               "2010-08-20 al 2010-11-20",
               "gravitational",
               resultado1['total'],
               resultado1['media_types'].get('image', 0),
               resultado1['media_types'].get('video', 0),
               lista_copyrights
    ]
}

# Convertimos a DataFrame
df_final = pd.DataFrame(datos_resumen)
display(df_final.style.hide(axis="index"))

Concepto,Valor
Alumno,Aguilera Yañez Mariana
Intervalo de fechas,2010-08-20 al 2010-11-20
Palabra buscada,gravitational
Total de Resultados,7
Media Type Imagen,7
Media Type Videos,0
Copyrights,"Neil Fleming: 1, Russell Croman: 1"


*   Alvarado Arce Axel Eduardo --> ***retrograde***


In [ ]:
df_Eduardo = get_apod_dataframe(
    api_key="S1ba65tHYH5KWWPUVwYfnGs3wpdXYf6TDMBQSQqk",
    start_date="2006-03-10",
    end_date="2010-07-19"
)

resultado1 = analizar_apod(df_Eduardo, "retrograde")

In [ ]:
import pandas as pd

# Convertir copyrights a lista de texto
lista_copyrights = ", ".join(
    [f" {nombre}: {cantidad}"
     for nombre, cantidad in resultado1['copyrights'].items()]
)


# Diccionario con los datos
datos_resumen = {
    "Concepto": ["Alumno","Intervalo de fechas", "Palabra buscada",
                 "Total de Resultados", "Media Type Imagen", "Media Type Videos",
                  "Copyrights"],
    "Valor": [ "Alvarado Arce Axel Eduardo",
               "2006-03-10 al 2010-07-19",
               "gravitational",
               resultado1['total'],
               resultado1['media_types'].get('image', 0),
               resultado1['media_types'].get('video', 0),
               lista_copyrights
    ]
}

# Convertimos a DataFrame
df_final = pd.DataFrame(datos_resumen)
display(df_final.style.hide(axis="index"))

Concepto,Valor
Alumno,Alvarado Arce Axel Eduardo
Intervalo de fechas,2006-03-10 al 2010-07-19
Palabra buscada,gravitational
Total de Resultados,5
Media Type Imagen,5
Media Type Videos,0
Copyrights,"Tunç Tezel: 2, Tunc Tezel: 1, Peter Wienerroither: 1, John Pane: 1"


*   Canseco Galván Tanivet --> ***supernova***

In [ ]:
df_Tanivet = get_apod_dataframe(
    api_key="S1ba65tHYH5KWWPUVwYfnGs3wpdXYf6TDMBQSQqk",
    start_date="2012-09-21",
    end_date="2013-05-7"
)

resultado1 = analizar_apod(df_Tanivet, "supernova")

In [ ]:
import pandas as pd

# Convertir copyrights a lista de texto
lista_copyrights = ", ".join(
    [f" {nombre}: {cantidad}"
     for nombre, cantidad in resultado1['copyrights'].items()]
)


# Diccionario con los datos
datos_resumen = {
    "Concepto": ["Alumno","Intervalo de fechas", "Palabra buscada",
                 "Total de Resultados", "Media Type Imagen", "Media Type Videos",
                  "Copyrights"],
    "Valor": [ "Canseco Galván Tanivet",
               "2012-09-21 al 2013-05-7",
               "gravitational",
               resultado1['total'],
               resultado1['media_types'].get('image', 0),
               resultado1['media_types'].get('video', 0),
               lista_copyrights
    ]
}

# Convertimos a DataFrame
df_final = pd.DataFrame(datos_resumen)
display(df_final.style.hide(axis="index"))

Concepto,Valor
Alumno,Canseco Galván Tanivet
Intervalo de fechas,2012-09-21 al 2013-05-7
Palabra buscada,gravitational
Total de Resultados,21
Media Type Imagen,21
Media Type Videos,0
Copyrights,"Martin Pugh: 2, Rogelio Bernal Andreo: 1, Don Goldman: 1, Joaquin Ferreiros: 1, Peter Tuthill (Sydney U.) & James Lloyd (Cornell): 1, Kfir Simon: 1, SSRO-South (S. Mazlin, J. Harvey, D. Verschatse, R. Gilbert) & Kevin Ivarsen (UNC/CTIO/PROMPT): 1, Dieter Willasch: 1, John Davis: 1, CXIELO Observatory: 1"


*   Garcia Soto Kevin --> ***nebula***

In [ ]:
df_Kevin = get_apod_dataframe(
    api_key="S1ba65tHYH5KWWPUVwYfnGs3wpdXYf6TDMBQSQqk",
    start_date="2010-06-28",
    end_date="2013-09-03"
)

resultado1 = analizar_apod(df_Kevin, "nebula")

In [ ]:
import pandas as pd

# Convertir copyrights a lista de texto
lista_copyrights = ", ".join(
    [f" {nombre}: {cantidad}"
     for nombre, cantidad in resultado1['copyrights'].items()]
)


# Diccionario con los datos
datos_resumen = {
    "Concepto": ["Alumno","Intervalo de fechas", "Palabra buscada",
                 "Total de Resultados", "Media Type Imagen", "Media Type Videos",
                  "Copyrights"],
    "Valor": [ "Garcia Soto Kevin",
               "2010-06-28 al 2013-09-03",
               "gravitational",
               resultado1['total'],
               resultado1['media_types'].get('image', 0),
               resultado1['media_types'].get('video', 0),
               lista_copyrights
    ]
}

# Convertimos a DataFrame
df_final = pd.DataFrame(datos_resumen)
display(df_final.style.hide(axis="index"))

Concepto,Valor
Alumno,Garcia Soto Kevin
Intervalo de fechas,2010-06-28 al 2013-09-03
Palabra buscada,gravitational
Total de Resultados,273
Media Type Imagen,270
Media Type Videos,3
Copyrights,"Adam Block: 12, Martin Pugh: 9, Don Goldman: 6, John Davis: 5, Rogelio Bernal Andreo: 5, Ken Crawford: 4, Rolf Geissinger: 4, Tony Hallas: 4, Glittering Lights: 4, Stephen Leshin: 4, Robert Gendler: 3, Astro Anarchy: 3, Ivan Eder: 3, Daniel López: 2, John P. Gleason: 2, Rogelio Bernal Andreo: 2, T. Rector: 2, Star Echoes: 2, Jerry Lodriguss (Catching the Light): 2, R Jay Gabany: 2, Babak Tafreshi: 2, Dieter Willasch (Astro-Cabinet): 2, Bob Franke: 2, Ignacio Diaz Bobillo: 2, Dieter Willasch: 2, Scott Rosen: 2, Yuri Beletsky: 2, Rogelio Bernal Andreo (Deep Sky Colors): 2, Nicol�s Villegas: 2, Peter Tuthill (Sydney U.) & James Lloyd (Cornell): 2, Fera Photography: 2, Bob Caton: 2, Máximo Ruiz: 2, Jaime Fernández: 1, Sergi Verdugo Martínez: 1, Garden by Jon Lomberg; Kite Aerial Photography by Pierre and Heidy Lesage: 1, Michael Sidonio: 1, Marcelo Salemme: 1, ESO: 1, Matthew T. Russell: 1, Jaime Fernandez: 1, Dave Jurasevich: 1, Ray Gralak: 1, Piotrek Sadowski: 1, Maurice Toet, Steve Loughran, Darren Jehan & Tim Jardine: 1, Derek Santiago: 1, Star Shadows Remote Observatory: 1, IAC: 1, Alessandro Falesiedi: 1, Neil Fleming: 1, Jean-Luc Dauvergne (Ciel et Espace); Music: Val�re Leroy & Sophie Huet (Space-Music): 1, Alex Cherney (Terrastro); Music: Redmann: 1, Leonardo Julio (Astronomia Pampeana): 1, Photopic Sky Survey: 1, Markus Noller (Deep Sky Images): 1, Mark Hanson: 1, Burrell Durrant Hifle Design & Direction Limited; Music Score: Timo Baker: 1, Geert Barentsen & Jorick Vink (Armagh Observatory) & the IPHAS Collaboration: 1, Jens Hackmann: 1, Brian Lula: 1, Brendan Alexander (Donegal Skies): 1, Hay Creek Observatory: 1, Manuel Fernández Suarez: 1, Daniel Verloop (Beursacademie): 1, Lóránd Fényes: 1, Larry Van Vleet: 1, Donegal Skies: 1, Brian Davis: 1, Leonardo Orazi: 1, Steve Cannistra: 1, Leonardo Julio & Carlos Milovic (Astronomia Pampeana): 1, Harel Boren: 1, Sergio Eguivar (Buenos Aires Skies): 1, Brno Univ. of Technology: 1, Osservatorio MTM: 1, West Mountain Observatory: 1, Yves Van den Broek: 1, Lori Allen, Xavier Koenig (Harvard-Smithsonian CfA) et al., JPL-Caltech, NASA: 1, Guillaume Blanchard: 1, Bill Snyder (Bill Snyder Photography): 1, Gimmi Ratto & Davide Bardini (Collecting Photons): 1, Jes�s Vargas (Astrogades) & Maritxu Poyal (Maritxu): 1, Frédéric Burgeot: 1, Fabian Neyer: 1, Mike Salway: 1, Star Shadows Remote Observatory: 1, Tom O'Donoghue: 1, Descubre Foundation, CAHA, OAUV, DSA, Vicent Peris (OAUV), Jack Harvey (SSRO), PixInsight: 1, Carlos Milovic, Hubble Legacy Archive, NASA: 1, Juan Carlos Casado: 1, Thierry: 1, Stefano Cancelli: 1, Kfir Simon: 1, Joaquin Ferreiros: 1, Nick Pavelchak: 1, Luis Argerich: 1, Nigel Sharp: 1, Ignacio de la Cueva Torregrosa: 1, CXIELO Observatory: 1, Terry Hancock: 1, Reinhold Wittich: 1, Wally Pacholka: 1, Bob Andersson: 1, L. Comolli, L. Fontana, G. Ghioldi & E. Sordini: 1, C�sar Blanco Gonz�lez: 1, Desert Hollow Observatory: 1, Bill Snyder: 1, Jose Francisco Hernandez (Altamira Observatory): 1, Fernando Cabrerizo: 1, Emanuele Colognato & Jim Wood: 1, Nick Martin: 1, Juan Lozano de Haro: 1, César Blanco González: 1, Jürg Alean: 1"


*   Hernandez Mareles Francisco Javier --> ***planetary***

In [ ]:
df_Javier = get_apod_dataframe(
    api_key="S1ba65tHYH5KWWPUVwYfnGs3wpdXYf6TDMBQSQqk",
    start_date="2017-01-15",
    end_date="2018-08-29"
)

resultado1 = analizar_apod(df_Javier, "planetary")

In [ ]:
import pandas as pd

# Convertir copyrights a lista de texto
lista_copyrights = ", ".join(
    [f" {nombre}: {cantidad}"
     for nombre, cantidad in resultado1['copyrights'].items()]
)


# Diccionario con los datos
datos_resumen = {
    "Concepto": ["Alumno","Intervalo de fechas", "Palabra buscada",
                 "Total de Resultados", "Media Type Imagen", "Media Type Videos",
                  "Copyrights"],
    "Valor": [ "Hernandez Mareles Francisco Javier",
               "2017-01-15 al 2018-08-29",
               "gravitational",
               resultado1['total'],
               resultado1['media_types'].get('image', 0),
               resultado1['media_types'].get('video', 0),
               lista_copyrights
    ]
}

# Convertimos a DataFrame
df_final = pd.DataFrame(datos_resumen)
display(df_final.style.hide(axis="index"))

Concepto,Valor
Alumno,Hernandez Mareles Francisco Javier
Intervalo de fechas,2017-01-15 al 2018-08-29
Palabra buscada,gravitational
Total de Resultados,29
Media Type Imagen,28
Media Type Videos,1
Copyrights,"Jes�s M.Vargas & Maritxu Poyal: 2, Raul Villaverde: 1, Adam Block: 1, Barry Riu: 1, Göran Strand: 1, Subaru, NAOJ: 1, Dave Jurasevich (Mt. Wilson Observatory): 1, Josep Drudis: 1, Astronomical Society of Linz: 1, Domingo Pestana & Raul Villaverde: 1, Robert Gendler: 1"


*   Rendon Cardoso Karla --> ***Geminids***

In [ ]:
df_Karla = get_apod_dataframe(
    api_key="S1ba65tHYH5KWWPUVwYfnGs3wpdXYf6TDMBQSQqk",
    start_date="2023-07-26",
    end_date="2026-04-02"
)

resultado1 = analizar_apod(df_Karla, "Geminids")

In [ ]:
import pandas as pd

# Convertir copyrights a lista de texto
lista_copyrights = ", ".join(
    [f" {nombre}: {cantidad}"
     for nombre, cantidad in resultado1['copyrights'].items()]
)


# Diccionario con los datos
datos_resumen = {
    "Concepto": ["Alumno","Intervalo de fechas", "Palabra buscada",
                 "Total de Resultados", "Media Type Imagen", "Media Type Videos",
                  "Copyrights"],
    "Valor": [ "Rendon Cardoso Karla",
               "2023-07-26 al 2026-04-02",
               "gravitational",
               resultado1['total'],
               resultado1['media_types'].get('image', 0),
               resultado1['media_types'].get('video', 0),
               lista_copyrights
    ]
}

# Convertimos a DataFrame
df_final = pd.DataFrame(datos_resumen)
display(df_final.style.hide(axis="index"))

Concepto,Valor
Alumno,Rendon Cardoso Karla
Intervalo de fechas,2023-07-26 al 2026-04-02
Palabra buscada,gravitational
Total de Resultados,6
Media Type Imagen,5
Media Type Videos,1
Copyrights,"William Ostling, Telescope Live: 1, Hongyang Luo: 1, Tomáš Slovinský: 1, Xu Chen: 1"


*   Suarez Urbano David Hiram --> ***Einstein***

In [ ]:
df_David = get_apod_dataframe(
    api_key="S1ba65tHYH5KWWPUVwYfnGs3wpdXYf6TDMBQSQqk",
    start_date="2009-09-08",
    end_date="2012-07-12"
)

resultado1 = analizar_apod(df_David, "Einstein")

In [ ]:
import pandas as pd

# Convertir copyrights a lista de texto
lista_copyrights = ", ".join(
    [f" {nombre}: {cantidad}"
     for nombre, cantidad in resultado1['copyrights'].items()]
)


# Diccionario con los datos
datos_resumen = {
    "Concepto": ["Alumno","Intervalo de fechas", "Palabra buscada",
                 "Total de Resultados", "Media Type Imagen", "Media Type Videos",
                  "Copyrights"],
    "Valor": ["Suarez Urbano David Hiram",
               "2009-09-08 al 2012-07-12",
               "gravitational",
               resultado1['total'],
               resultado1['media_types'].get('image', 0),
               resultado1['media_types'].get('video', 0),
               lista_copyrights
    ]
}

# Convertimos a DataFrame
df_final = pd.DataFrame(datos_resumen)
display(df_final.style.hide(axis="index"))

Concepto,Valor
Alumno,Suarez Urbano David Hiram
Intervalo de fechas,2009-09-08 al 2012-07-12
Palabra buscada,gravitational
Total de Resultados,5
Media Type Imagen,4
Media Type Videos,1
Copyrights,"J. Rhoads: 1, Andrew Truscott & Randall Hulet (Rice U.): 1"


**CONCLUSIONES GENERALES:**

El análisis que se hizo en la API APOD de la NASA muestra contenido astronómico y multimedia.
Los resultados obtenidos dependen de la palabra clave y el parametro del rango de fechas, aparecieron imagenes, videos, diferentes autores. Demostrando que la API es una colección dinamica de contenido cientifico y también divulgativo.

Palabras como *nebula*, *supernova*, *planetary*, tuvieron más resultados porque son conceptos comunes en astronomía, generando mucho contenido visual, es por eso que se generaron más resultados de media y múltiples autores.
Mientras palabras como *retrograde*, *einstein*, *gravitational*, generarón menos resultados ya que son conceptos más específicos .

El tipo de resultados en el tipo de media, en algunos casos se obtuvieron imágenes en otroa videos o ambos, esto depende del fenómenos astronómico ya que muchos de ellos se explican mejor mediante animaciones.
El campo de copyrights evidencia que la APOD utiliza contenido de astrónomos, fotógrafos, observatorios, agencias, funcionando también como una plataforma de difusión cientifica colaboratuiva.



